In [ ]:
import os
import random
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
from termcolor import colored, cprint

from inspect_ai.log import read_eval_log

In [ ]:
root_dir = os.path.join("..", "outputs", "first_epoch")
eval_dir = "text"

# Prepare data

In [ ]:
def result_list_to_df(results):
    entries = []
    # Expand elements in `results` to one Dict per sample
    for res in results:
        metadata = {k: res[k] for k in ("model_name", "experiment", "question_type")}
        for samples_cur, corr in zip([res["samples_correct"], res["samples_wrong"]], (True, False)):
            for smpl in samples_cur:
                sample_data = {
                    "id": smpl.id,
                    "question": smpl.input,
                    "score_answer": smpl.score.answer,  # For sanity checking. Should match 'Model answer'
                    "model_answer": "\n".join(c.text for c in smpl.messages[1].content if c.type=="text"),
                    "model_reasoning": "\n".join(c.reasoning for c in smpl.messages[1].content if c.type=="reasoning"),
                    "grader_score_explanation": smpl.score.explanation,
                    "grader_score": corr,
                    "target": smpl.target,
                }            
                entries.append(dict(**metadata, **sample_data))
    return pd.DataFrame(entries)

In [ ]:
# For each model, draw `n_sample` correctly and `n_sample` wrongly answered questions
n_sample = 1

results = []
for model_dir in os.listdir(os.path.join(root_dir, eval_dir)):
    model_path = os.path.join(root_dir, eval_dir, model_dir)
    print("Processing: " + colored(model_dir, "cyan"))
  
    # Read evaluation log
    try:
        eval_files = [f for f in os.listdir(model_path) if f.endswith(".eval")]
        log_file = os.path.join(model_path, eval_files[0])
        log = read_eval_log(log_file)
    except Exception as e:
        print(colored(f"\tError reading log for {model_dir}: {e}", "red"))
        continue

    n_models = len(log.stats.model_usage.keys())
    
    # Compute number valid samples
    idx_valid = [i for i, sample in enumerate(log.samples) if sample.score is not None and sample.error is None]
    idx_correct = [i for i, sample in enumerate(log.samples) if i in idx_valid and sample.score.value in [1, True, "C"]]
    idx_wrong = [i for i, sample in enumerate(log.samples) if i in idx_valid and sample.score.value not in [1, True, "C"]]

    idx_sample_correct = random.sample(idx_correct, min(n_sample, len(idx_correct)))
    idx_sample_wrong = random.sample(idx_wrong, min(n_sample, len(idx_wrong)))

    samples_correct = [log.samples[i] for i in idx_sample_correct]
    samples_wrong = [log.samples[i] for i in idx_sample_wrong]
    
    
    results.append({
        "model_name": model_dir.split("_")[0],
        "experiment": root_dir.split("/")[-1],
        "num_valid_samples": len(idx_valid),
        "num_correct_samples": len(idx_correct),
        "num_wrong_samples": len(idx_wrong),
        "samples_correct": samples_correct,
        "samples_wrong": samples_wrong,
        "question_type": eval_dir,
    })

In [ ]:
results_df = result_list_to_df(results)

# Human grading of model answers

In [ ]:
for i, (idx, row) in enumerate(results_df.sample(frac=1).iterrows()):
    print(f"Question {i+1} of {len(results_df)}:")
    print(row.question)
    print("\nModel answer:")
    print(row.model_answer)
    print("\nCorrect answer:")
    print(row.target)
    if row.model_answer != row.score_answer:
        print("\nScore answer:")
        print(row.score_answer)
    # print("\nModel reasoning:")
    # print(row.model_reasoning)
    inp = ""
    while inp.lower() not in ("y", "n"):
        inp = input("\nIs the model answer correct [y/n]?\n")
    results_df.loc[idx, "human_score"] = (inp == "y")
    print("\n---------------------------------------------\n")

# Human evaluation of grader

In [ ]:
results_df["human_and_grader_scores_match"] = (results_df.grader_score == results_df.human_score)

In [ ]:
for i, (idx, row) in enumerate(results_df.iterrows()):
    print(f"Question {i+1} of {len(results_df)}:")
    print(row.question)
    print("\nModel answer:")
    print(row.model_answer)
    print("\nCorrect answer:")
    print(row.target)
    if row.model_answer != row.score_answer:
        print("\nScore answer:")
        print(row.score_answer)
    # print("\nModel reasoning:")
    # print(row.model_reasoning)
    print("\nGrader score:")
    print(row.grader_score)

    if row.human_and_grader_scores_match:
        print(f"\n Human and grader agree that model answer is {'*CORRECT*' if row.human_score else '*FALSE*'}")
    else:
        print("\n Human and grader do not agree - human says the answer is "
              f"{'*CORRECT*' if row.human_score else '*FALSE*'}, grader says "
              f"{'*CORRECT*' if row.grader_score else '*FALSE*'}")
    
    print("\nGrader explanation:")
    print(row.grader_score_explanation)
            
    inp = ""
    while inp.lower() not in ("y", "n"):
        inp = input("\nDo you agree with the grader explanation?\n")
    results_df.loc[idx, "human_agrees_with_grader_explanation"] = (inp == "y")

# Export results

In [ ]:
results_df.to_csv(f"validation_{eval_dir}.csv")